# Lesson 27b: ANOVA in Multiple Linear Regression

Is a model's improvement real, or could chance explain it? In this notebook we:

1. **Recap: four models.** Refit the hybrid-car models from Lessons 26–27.
2. **Sums of squares.** Split the variation in MSRP into SST = SSR + SSE.
3. **Permutation test.** Shuffle MSRP to see what F looks like when the features don't help.
4. **ANOVA table.** Get the p-value from the F-distribution.
5. **One-way ANOVA.** Compare mean MSRP across car classes.
6. **Partial F-test.** Does car class help beyond acceleration and mpg?
7. **Noise feature.** Watch the test say "no".

Section names match the **Jupyter demo** tags on the slides.

[Acknowledgments Page](https://ds100.org/fa23/acks/)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

rng = np.random.default_rng(3022)   # fixed seed so results match the slides

## 1. Recap: four models

Same data and features as Lesson 27: acceleration, mpg, their transformations, and one-hot encoded car class (drop-first, so one class is the baseline).

In [ ]:
hybrid = pd.read_csv('data/hybrid.csv')
y = hybrid['msrp'].to_numpy()
n = len(y)

hybrid['acc_sq'] = hybrid['acceleration'] ** 2
hybrid['mpg_sqrt'] = np.sqrt(hybrid['mpg'])

ohe = OneHotEncoder(drop='first')
ohe.fit(hybrid[['class']])
class_cols = pd.DataFrame(ohe.transform(hybrid[['class']]).toarray(),
                          columns=ohe.get_feature_names_out(), index=hybrid.index)

quant = ['acceleration', 'mpg', 'acc_sq', 'mpg_sqrt']
feature_sets = {
    'SLR':    hybrid[['acceleration']],
    'MLR V1': hybrid[['acceleration', 'mpg']],
    'MLR V2': hybrid[quant],
    'MLR V3': hybrid[quant].join(class_cols),
}
n

A helper that fits a model and returns the three sums of squares. We'll use it throughout.

In [ ]:
def sums_of_squares(X, y):
    """Fit least squares (with intercept) and return SST, SSR, SSE and p."""
    model = LinearRegression().fit(X, y)
    y_hat = model.predict(X)
    sst = np.sum((y - y.mean()) ** 2)
    ssr = np.sum((y_hat - y.mean()) ** 2)
    sse = np.sum((y - y_hat) ** 2)
    return {'SST': sst, 'SSR': ssr, 'SSE': sse, 'p': X.shape[1]}

rows = []
for name, X in feature_sets.items():
    ss = sums_of_squares(X, y)
    p = ss['p']
    r2 = ss['SSR'] / ss['SST']
    rows.append({'model': name, 'p': p,
                 'RMSE': np.sqrt(ss['SSE'] / n),
                 'R2': r2,
                 'adj_R2': 1 - (1 - r2) * (n - 1) / (n - p - 1)})
pd.DataFrame(rows).round(3)

Every added feature raises R². Next we ask: is each improvement more than chance?

## 2. Sums of squares

For least squares with an intercept, **SST = SSR + SSE**, and **R² = SSR / SST**.

In [ ]:
ss_table = pd.DataFrame({name: sums_of_squares(X, y) for name, X in feature_sets.items()}).T
ss_table['SSR + SSE'] = ss_table['SSR'] + ss_table['SSE']
ss_table['R2'] = ss_table['SSR'] / ss_table['SST']
(ss_table[['SST', 'SSR', 'SSE', 'SSR + SSE', 'R2']] / [1e6, 1e6, 1e6, 1e6, 1]).round(3)   # sums of squares in millions of $^2

SST is the same for every model (it doesn't depend on the features), and SSR + SSE matches it exactly. Adding features moves variation from SSE into SSR.

**Check:** R² is also Var(ŷ) / Var(y), the coefficient of determination from Lesson 24.

In [ ]:
y_hat_v2 = LinearRegression().fit(feature_sets['MLR V2'], y).predict(feature_sets['MLR V2'])
np.var(y_hat_v2) / np.var(y), ss_table.loc['MLR V2', 'R2']

## 3. Permutation test

The overall F-test asks: H₀: θ₁ = … = θₚ = 0 (the features don't help) vs. Hₐ: at least one θⱼ ≠ 0.

$$F = \frac{SSR / p}{SSE / (n - p - 1)} = \frac{MSR}{MSE}$$

Note: in ANOVA, MSE divides by n − p − 1, not n.

In [ ]:
def overall_F(X, y):
    ss = sums_of_squares(X, y)
    p = ss['p']
    return (ss['SSR'] / p) / (ss['SSE'] / (n - p - 1))

X_v2 = feature_sets['MLR V2']
F_obs = overall_F(X_v2, y)
F_obs

Shuffling MSRP breaks any link between the features and price, so H₀ is true for the shuffled data. Collect F from 2,000 shuffles.

In [ ]:
hybrid['noise'] = rng.normal(size=n)   # used in Section 7; drawn here so the seed sequence matches the slides

shuffled_F = np.array([overall_F(X_v2, rng.permutation(y)) for _ in range(2000)])

print('largest shuffled F:', shuffled_F.max().round(2))
print('permutation p-value:', np.mean(shuffled_F >= F_obs))

In [ ]:
p, df2 = 4, n - 4 - 1
grid = np.linspace(0.01, 7, 300)
plt.hist(shuffled_F, bins=40, density=True, alpha=0.6, label='F from 2,000 shuffles')
plt.plot(grid, stats.f.pdf(grid, p, df2), lw=2, label=f'F-distribution ({p}, {df2})')
plt.xlabel('F statistic'); plt.ylabel('density')
plt.title(f'Observed F = {F_obs:.0f} is far off the chart')
plt.legend();

None of the shuffles comes close to the observed F, so the p-value is less than 1/2,000.

Notice the F-distribution curve matches the shuffled F's closely. That's the shortcut in the next section.

## 4. ANOVA table

If the conditions hold (independent observations, constant spread, normal noise), then under H₀, F follows an F-distribution with (p, n − p − 1) degrees of freedom.

In [ ]:
def anova_table(X, y):
    ss = sums_of_squares(X, y)
    p = ss['p']
    msr, mse = ss['SSR'] / p, ss['SSE'] / (n - p - 1)
    F = msr / mse
    return pd.DataFrame({
        'df':             [p, n - p - 1, n - 1],
        'sum_sq':         [ss['SSR'], ss['SSE'], ss['SST']],
        'mean_sq':        [msr, mse, np.nan],
        'F':              [F, np.nan, np.nan],
        'p_value':        [stats.f.sf(F, p, n - p - 1), np.nan, np.nan],
    }, index=['Model', 'Residual', 'Total'])

anova_table(X_v2, y)

`stats.f.sf(F, p, n - p - 1)` is the area to the right of F under the F-distribution: the p-value.

**Check:** the R² version of F gives the same number.

In [ ]:
r2 = ss_table.loc['MLR V2', 'R2']
(r2 / 4) / ((1 - r2) / (n - 4 - 1)), F_obs

## 5. One-way ANOVA

Does mean MSRP differ across car classes? H₀: all class means are equal.

In [ ]:
order = hybrid.groupby('class')['msrp'].mean().sort_values().index
plt.figure(figsize=(9, 4))
sns.boxplot(data=hybrid, x='class', y='msrp', order=order, color='lightsteelblue')
sns.stripplot(data=hybrid, x='class', y='msrp', order=order, size=3, color='k', alpha=0.5)
plt.axhline(y.mean(), ls='--', color='goldenrod', label='overall mean')
plt.legend();

In [ ]:
hybrid.groupby('class')['msrp'].agg(['count', 'mean', 'std']).round(0).sort_values('mean')

One-way ANOVA is the overall F-test for a regression on the one-hot class columns alone. Each prediction is that class's mean, so SSR = between-class variation and SSE = within-class variation.

In [ ]:
anova_table(class_cols, y)

Same F and p-value as scipy's built-in one-way ANOVA:

In [ ]:
groups = [g['msrp'].to_numpy() for _, g in hybrid.groupby('class')]
stats.f_oneway(*groups)

**Caution:** the class SDs range from about \$800 to about \$25,000, so the equal-spread condition is doubtful. A permutation test that shuffles the class labels doesn't need it:

In [ ]:
F_class = overall_F(class_cols, y)
shuffled = np.array([overall_F(class_cols, rng.permutation(y)) for _ in range(2000)])
print('observed F:', F_class.round(2), '  permutation p-value:', np.mean(shuffled >= F_class))

## 6. Partial F-test

Reduced model: MLR V2 (4 quantitative features). Full model: MLR V3 (the same 4 + 6 class columns). They're **nested**.

$$F = \frac{(SSE_{reduced} - SSE_{full}) / q}{SSE_{full} / (n - p_{full} - 1)}$$

In [ ]:
def partial_F(X_reduced, X_full, y):
    r, f = sums_of_squares(X_reduced, y), sums_of_squares(X_full, y)
    q = f['p'] - r['p']
    df2 = n - f['p'] - 1
    F = ((r['SSE'] - f['SSE']) / q) / (f['SSE'] / df2)
    return pd.Series({'SSE_reduced': r['SSE'], 'SSE_full': f['SSE'], 'q': q, 'df2': df2,
                      'F': F, 'p_value': stats.f.sf(F, q, df2)})

partial_F(feature_sets['MLR V2'], feature_sets['MLR V3'], y)

F ≈ 8.8 with p ≈ 3 × 10⁻⁸: strong evidence that car class is associated with MSRP among cars with the same acceleration and mpg. This agrees with adjusted R² rising from 0.63 to 0.72.

## 7. Noise feature

Add a column of random noise to MLR V2. R² still goes up, but the test should say "no".

In [ ]:
X_noise = hybrid[quant + ['noise']]
r2_v2 = ss_table.loc['MLR V2', 'R2']
ss_noise = sums_of_squares(X_noise, y)
r2_noise = ss_noise['SSR'] / ss_noise['SST']
adj = lambda r2, p: 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f'R^2:          {r2_v2:.4f} -> {r2_noise:.4f}')
print(f'adjusted R^2: {adj(r2_v2, 4):.4f} -> {adj(r2_noise, 5):.4f}')
partial_F(X_v2, X_noise, y)

**Try it:** rerun the test with 1,000 different noise columns. About 5% of p-values fall below 0.05 just by chance (Type I errors).

In [ ]:
p_values = np.array([partial_F(X_v2, X_v2.assign(noise=rng.normal(size=n)), y)['p_value'] for _ in range(1000)])
print('fraction of p-values < 0.05:', np.mean(p_values < 0.05))
plt.hist(p_values, bins=20, edgecolor='white')
plt.xlabel('p-value'); plt.ylabel('count'); plt.title('Partial F-test p-values for useless features');

## Optional: check with statsmodels

If statsmodels is installed, its ANOVA functions give the same numbers.

In [ ]:
try:
    import statsmodels.api as sm
    full = sm.OLS(y, sm.add_constant(feature_sets['MLR V3'])).fit()
    reduced = sm.OLS(y, sm.add_constant(feature_sets['MLR V2'])).fit()
    print('overall F (V3):', round(full.fvalue, 2))
    print(sm.stats.anova_lm(reduced, full))
except ImportError:
    print('statsmodels not installed; skip this cell')